# Vision Practice 02. ViT CIFAR-10 코드 학습 — 시험 대비 실습본

- 원본: `vision/02_ViT_CIFAR10.ipynb`
- 핵심 빈칸 수: **4개**
- `## 정답 입력` 셀을 직접 구현한 뒤 과목별 정답·해설지와 비교하세요.
- API key, 외부 서버 주소, 대용량 데이터 경로는 자신의 환경에 맞게 설정하세요.


# **Vision Transformer with CIFAR-10 — 패치 임베딩부터 분류까지**

**Vision Transformer(ViT)** 실습을 목표로 합니다.  
CNN과 달리 ViT는 이미지를 **패치(patch) 시퀀스**로 바꾼 뒤, NLP의 Transformer Encoder와 거의 같은 방식으로 처리합니다.

## 학습 목표
- 이미지를 **패치 토큰**으로 바꾸는 과정(= Patch Embedding)을 이해한다.
- **[CLS] 토큰 + Positional Embedding** 이 왜 필요한지 설명할 수 있다.
- Transformer Encoder의 핵심 구성(**Pre-LN / MHSA / FFN / Residual**)을 코드에서 찾아 읽을 수 있다.
- 학습 후 **오분류(실패) 샘플**을 통해 모델의 한계를 분석한다.

> 권장 흐름: (1) 데이터/전처리 → (2) ViT 구성 → (3) 학습/평가 → (4) 오분류 분석


---
## 0) Setup (환경 준비)

- **목적:** 실습에 필요한 패키지를 로드하고, 재현 가능한 실험을 위해 시드를 고정합니다.
- **관찰 포인트**
  - 실습에 필요한 유틸 패키지(`einops`, `torchinfo` 등) 준비
  - GPU 사용 여부(`cuda`) 확인

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(pkg_name: str, import_name=None):
    """
    패키지 설치 여부를 확인하고, 설치되어 있지 않으면 설치합니다.
    """
    name = import_name or pkg_name
    # 패키지가 설치되어 있는지 확인
    if importlib.util.find_spec(name) is None:
        print(f"[install] {pkg_name} 라이브러리를 설치 중입니다... (import name: {name})")
        try:
            # -q 옵션을 추가하여 설치 과정을 간결하게 유지할 수 있습니다.
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg_name])
            print(f"[success] {pkg_name} 설치 완료.")
        except subprocess.CalledProcessError as e:
            print(f"[error] {pkg_name} 설치 실패: {e}")
    else:
        print(f"[ok] {pkg_name} 이미 설치되어 있습니다.")

# 설치가 필요한 패키지 리스트 (패키지명, 임포트명)
# 임포트명이 패키지명과 다른 경우 튜플로 지정합니다.
packages = [
    ("einops", "einops"),
    ("torchinfo", "torchinfo")
]

# 루프를 돌며 확인 및 설치
for pkg, imp in packages:
    ensure_package(pkg, imp)

In [ ]:
import torch
from torch import nn
from torch import nn, einsum
import torch.nn.functional as F  # 함수형 API(F): activation/loss 등
from torch import optim

from einops import rearrange, repeat  # ViT에서 자주 쓰는 텐서 재배열(rearrange)·복제(repeat)
from einops.layers.torch import Rearrange
import numpy as np
import torchvision  # 데이터/변환(torchvision) 사용 (CIFAR-10 포함)
import time
from torchinfo import summary

print('torch:', torch.__version__)  # torch 버전 출력
print('torchvision:', torchvision.__version__)  # torchvision 버전 출력
print('cuda available:', torch.cuda.is_available())  # GPU 사용 가능 여부
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))  # GPU 이름 출력

try:
    get_ipython().system('nvidia-smi -L')  # GPU 목록 확인(가능한 경우)
except Exception as e:
    print('nvidia-smi not available:', e)  # nvidia-smi가 없는 환경이면 무시

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


---
## 1) 데이터 로딩 & 전처리

- **목적:** CIFAR-10(컬러 32×32)를 로드하고, 모델 입력 텐서 형태를 확인합니다.
- **관찰 포인트**
  - 입력 텐서 shape: `B×C×H×W` (CIFAR-10는 `C=3`)
  - 학습/테스트 로더 구성과 배치 단위 학습의 의미


In [ ]:
import os
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

# CIFAR-10에서 널리 쓰는 normalize 값
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)  # 채널별 평균(R,G,B)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)  # 채널별 표준편차(R,G,B)

train_tfms = T.Compose([  # train transform(augmentation 포함)
    T.RandomCrop(32, padding=4),      # 32x32를 padding 후 random crop
    T.RandomHorizontalFlip(),         # 좌우 반전
    T.ToTensor(),                     # PIL -> torch tensor (C,H,W)
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),  # 정규화
])

test_tfms = T.Compose([  # test transform(augmentation 없음)
    T.ToTensor(),                     # 텐서 변환
    T.Normalize(CIFAR10_MEAN, CIFAR10_STD),  # 정규화
])

data_root = './data'  # 데이터 저장 경로
train_set = torchvision.datasets.CIFAR10(root=data_root, train=True, download=True, transform=train_tfms)  # train set
test_set  = torchvision.datasets.CIFAR10(root=data_root, train=False, download=True, transform=test_tfms)  # test set

class_names = train_set.classes  # 클래스 이름 목록
num_classes = len(class_names)   # 클래스 개수(10)
print('classes:', class_names)   # 클래스 출력

# 이미지 사이즈 출력
x0, y0 = train_set[0]  # 한 샘플 로드(이미 transform 적용된 텐서)
print('Sample image tensor shape (C,H,W):', tuple(x0.shape))  # (3,32,32)

batch_size = 256  # 배치 크기(A10이면 보통 여유)
num_workers = min(8, os.cpu_count() or 2)  # dataloader worker 수(환경에 맞게)

train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)  # train loader
test_loader  = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)  # test loader

print('Total batch size (train: %d, test: %d)' % (len(train_loader), len(test_loader)))  # 배치 개수 확인


In [ ]:
# ## 정답 입력
# Drill 1: 1) 데이터 로딩 & 전처리
# 원본 Cell 006의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 2) ViT 모델 구성 개요

ViT는 크게 아래 3단계로 이해하면 읽기 쉬워집니다.

1. **Patch Embedding**  
   이미지(`H×W`)를 `P×P` 패치로 잘라 **토큰 시퀀스**로 바꾸고, 각 패치를 `dim` 차원으로 임베딩합니다.

2. **Transformer Encoder (× depth)**  
   토큰 시퀀스에 대해 반복적으로  
   **(Pre-LN → Multi-Head Self-Attention → Residual) + (Pre-LN → FFN → Residual)** 를 수행합니다.

3. **Classification Head**  
   보통 `[CLS]` 토큰(또는 평균 풀링)을 사용해 최종 분류 로짓을 출력합니다.

> 아래 코드에서 `dim / depth / heads / mlp_dim` 이 무엇을 의미하는지 주석과 함께 확인해 보세요.


In [ ]:
# ## 정답 입력
# Drill 2: 2) ViT 모델 구성 개요
# 원본 Cell 008의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 2-1) ViTConfig + ViT(모델 본체) 구현

- **목적:** ViT의 핵심 구성 요소(패치 임베딩, CLS 토큰, 위치 임베딩, Transformer Encoder 블록, 분류 헤드)를 **코드로 직접 확인**합니다.
- **관찰 포인트**
  - `image_size`, `patch_size` → 패치 개수(`num_patches`)와 토큰 시퀀스 길이가 어떻게 결정되는지
  - `depth`(블록 개수), `heads`(멀티헤드 수), `dim`(토큰 임베딩 차원)이 **연산량/표현력**에 어떤 영향을 주는지
  - **Pre-LN + Residual** 구조가 어디에 적용되는지(안정적인 학습을 위한 표준 패턴)


In [ ]:
# ## 정답 입력
# Drill 3: 2-1) ViTConfig + ViT(모델 본체) 구현
# 원본 Cell 010의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 2-2) ViT 하이퍼파라미터 설정 (vit_cfg)

- **목적:** ViT의 구조를 결정하는 설정값을 한 곳에서 정의하고, 아래 `ViT(vit_cfg)`가 **어떤 형태의 모델**을 만드는지 연결해 봅니다.
- **관찰 포인트**
  - `patch_size`가 작아질수록 토큰 수가 늘어 **Self-Attention 비용**이 증가합니다.
  - `depth/heads/dim`은 성능뿐 아니라 **학습 시간·메모리**에도 직접 영향을 줍니다.
  - CIFAR-10(32×32, RGB)에서도 과도한 설정은 **학습 불안정/과적합**을 유발할 수 있습니다.


In [ ]:
# ViT 설정을 한 곳에 모아두면(=config) 모델 구조를 한눈에 파악하기 쉽습니다.
vit_cfg = ViTConfig(
        image_size=32,      # CIFAR-10: 32x32
    patch_size=4,       # 4x4 패치 -> (32/4)^2 = 64개 패치 토큰
        channels=3,         # CIFAR-10은 컬러(3채널)
    dim=64,             # 토큰 임베딩 차원
    depth=6,            # Encoder 블록 수
    heads=4,            # MHSA head 수
    dim_head=64,        # head당 Q/K/V 차원
    mlp_dim=128,        # FFN 중간 차원
    num_classes=10,     # 0~9
    pool="cls",         # CLS 토큰으로 분류
    dropout=0.0,        # Encoder dropout
    emb_dropout=0.0,    # 임베딩 dropout
)

model = ViT(vit_cfg).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003)    # Momentum + RMSProp


## 4) 체크포인트 저장/이어학습(Resume)

- **목적:** 학습 결과를 `ckpt_path`로 저장하고, 파일이 있으면 **이어서 추가 학습**합니다.
- **관찰 포인트**
  - 체크포인트가 있을 때: `start_epoch`부터 이어서 **추가 10 epoch**
  - 체크포인트가 없을 때: 처음부터 **20 epoch**


In [ ]:
# 체크포인트(ckpt) 저장/로드
# - ckpt 파일이 있으면: 이어서 추가 5 epoch 학습(resume)
# - ckpt 파일이 없으면: 처음부터 10 epoch 학습

ckpt_path = "vit_cifar10.pth"  # 필요시 파일명/경로만 바꿔서 사용하세요.
start_epoch = 0
last_acc = 0.0  # 체크포인트가 있을 때 로드되는 이전 test accuracy

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    start_epoch = int(ckpt.get("epoch", 0))
    last_acc = float(ckpt.get("test_acc", 0.0))
    print(f"[ckpt] Loaded: {ckpt_path} (epoch={start_epoch}, test_acc={last_acc*100:.2f}%)")
    extra_epochs = 10
else:
    print(f"[ckpt] Not found: {ckpt_path} -> train from scratch")
    extra_epochs = 20


# best_acc: 지금까지 저장된 체크포인트 중 최고 정확도(업데이트 기준)
best_acc = last_acc

target_epoch = start_epoch + extra_epochs
print(f"[plan] Train epochs: {start_epoch+1} .. {target_epoch} (total {extra_epochs} epoch(s))")


---
## 3) 모델 구조 요약 & 파라미터 수 확인

- **목적**: 레이어 구성을 빠르게 보고, 학습 가능한 파라미터 규모를 숫자로 확인합니다.
- **관찰 포인트:**
  - Patch Embedding이 만들어내는 토큰 수(= 패치 개수 + 1[CLS])
  - Encoder 블록이 `depth`만큼 반복되는지
  - 마지막 분류 헤드(Linear)가 어디에 붙는지
  - `dim/depth/heads`를 키우면 파라미터/연산량이 어떻게 커지는지

In [ ]:
# (1) 모델 구조 요약: 레이어와 텐서 흐름을 빠르게 확인
# torchinfo.summary는 입력 텐서 크기를 알면 훨씬 정확한 구조를 출력합니다.
_summary = summary(model, input_size=(1, 3, 32, 32), device=str(device))
print(_summary)

# (2) 학습 가능한 파라미터 수: 모델 '규모'를 숫자로 확인
def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Trainable parameters:", count_parameters(model))


---
## 4) Training (학습 루프)

- **목적:** ViT를 CIFAR-10 학습 데이터로 학습시켜, *토큰 기반 분류가 실제로 동작*하는 것을 확인합니다.
- **관찰 포인트:**
  - `model.train()` 상태에서 **드롭아웃/정규화**가 어떻게 동작하는지
  - 손실(loss)이 줄어드는지(학습이 되는지)



In [ ]:
def train_epoch(model, optimizer, data_loader, loss_history):
    """한 epoch 학습을 수행하고, 진행 상황을 epoch당 '딱 5번'만 출력합니다."""
    total_samples = len(data_loader.dataset)
    model.train()

    # epoch당 출력 5회(0%, 25%, 50%, 75%, 마지막)로 고정
    n_steps = len(data_loader)
    log_steps = {
        0,
        max(0, n_steps // 4),
        max(0, n_steps // 2),
        max(0, (3 * n_steps) // 4),
        n_steps - 1,
    }

    for i, (data, target) in enumerate(data_loader):
        data = data.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        output = F.log_softmax(model(data), dim=1)  # Softmax
        loss = F.nll_loss(output, target)           # Negative Log Likelihood
        # nn.CrossEntropyLoss(model(data), target)
        loss.backward()
        optimizer.step()

        if i in log_steps:
            seen = min((i + 1) * len(data), total_samples)
            print(f"[{seen:5d}/{total_samples:5d} ({100.0 * (i+1) / n_steps:5.1f}%)]  Loss: {loss.item():.4f}")
            loss_history.append(loss.item())

In [ ]:
def evaluate(model, data_loader, loss_history):
    """test_loader로 1회 평가하고 (loss, acc)를 반환합니다."""
    model.eval()

    total_samples = len(data_loader.dataset)
    correct_samples = 0
    total_loss = 0.0

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            output = F.log_softmax(model(data), dim=1)
            loss = F.nll_loss(output, target, reduction='sum')
            _, pred = torch.max(output, dim=1)

            total_loss += float(loss.item())
            correct_samples += int(pred.eq(target).sum().item())

    avg_loss = total_loss / total_samples
    acc = correct_samples / total_samples

    loss_history.append(avg_loss)
    print(
        f"Average test loss: {avg_loss:.4f}  "
        f"Accuracy: {correct_samples:5d}/{total_samples:5d} ({acc*100:.2f}%)\n"
    )
    return avg_loss, acc


> 참고: CPU로 실행하면 epoch 당 시간이 꽤 걸릴 수 있습니다.  
> 가능하면 GPU 환경에서 실행하는 것을 권장합니다. (CIFAR-10는 작지만, Transformer는 토큰 연산이 많습니다.)


---
## 5) Training (학습)

- **목적:** 학습 데이터로 파라미터를 업데이트합니다.
- **관찰 포인트:**
  - `model.train()` 모드에서 Dropout/BN 등이 어떻게 동작하는지
  - Loss가 epoch이 진행되며 감소하는지(과적합/학습 실패 징후 관찰)
  - 매 epoch 평가 후, **정확도가 개선된 경우에만(best) 체크포인트를 업데이트**합니다.

In [ ]:
# ckpt 기준으로 학습 epoch 수가 자동 결정됩니다.
# - ckpt가 있으면(start_epoch>0): 이어서 추가 10 epoch
# - ckpt가 없으면(start_epoch=0): 처음부터 20 epoch

start_time = time.time()

train_loss_history = []
test_loss_history = []

for epoch in range(start_epoch + 1, target_epoch + 1):
    print(f"Epoch: {epoch}/{target_epoch}")

    # 학습
    train_epoch(model, optimizer, train_loader, train_loss_history)

    # 매 epoch마다 test set으로 1회 평가(accuracy 출력)
    test_loss, test_acc = evaluate(model, test_loader, test_loss_history)

    # 정확도가 '개선'된 경우에만 체크포인트를 업데이트합니다.
    # - 이렇게 하면 가장 좋은(best) 모델만 ckpt_path에 남습니다.
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(
            {
                "epoch": epoch,
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "test_acc": test_acc,
                "test_loss": test_loss,
                "vit_cfg": getattr(vit_cfg, "__dict__", dict(vit_cfg=repr(vit_cfg))),
            },
            ckpt_path,
        )
        print(f"[ckpt] Updated(best): {ckpt_path} (epoch={epoch}, best_acc={best_acc*100:.2f}%)\n")
    else:
        print(f"[ckpt] Skip (not improved): best_acc={best_acc*100:.2f}% (this={test_acc*100:.2f}%)\n")

print("Execution time:", "{:5.2f}".format(time.time() - start_time), "seconds")


---
## 6) Testing (Attention)

- **목적:** ViT의 예측이 어떤 **패치(토큰)** 정보에 의존하는지, CLS 토큰의 attention을 통해 직관적으로 확인합니다.
- **관찰 포인트:**
  - CLS 토큰이 **어떤 위치(패치)** 에 강하게 attention을 주는지
  - layer/head를 바꾸면 attention 패턴이 어떻게 달라지는지
  - (선택) **정답/오답 샘플**에서 attention이 얼마나 설득력 있게 보이는지

In [ ]:
# ## 정답 입력
# Drill 4: 6) Testing (Attention)
# 원본 Cell 024의 핵심 코드를 직접 작성하세요.
# 과목별 정답·해설지에는 원본 코드와 출제 의도가 있습니다.

pass


---
## 7) Confusion Matrix로 오분류 패턴 보기

- **목적:** 단순 정확도(accuracy)만으로는 보이지 않는 **클래스별 오분류 경향**을 확인합니다.
- **관찰 포인트:**
  - 어떤 클래스 쌍(예: `cat↔dog`, `automobile↔truck`, `deer↔horse`)이 자주 헷갈리는지
  - 특정 클래스에서만 오분류가 집중되는지(데이터/모델 편향 가능성)
  - 이후 `오분류 샘플 시각화`와 함께 보면 “왜” 틀렸는지 해석이 쉬워집니다.


In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

def plot_confusion_matrix(model, data_loader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)
            output = model(data)  # 입력 배치에 대한 모델 forward(로짓 출력)
            _, preds = torch.max(output, dim=1)   # preds = maximum indices
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(target.cpu().numpy())

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)

    plt.ylabel('Actual')
    plt.title('Confusion Matrix')
    plt.show()

plot_confusion_matrix(model, test_loader)


---
## 8) 오분류(실패) 샘플만 시각화하기

- **목적:** 단순히 "맞춘 샘플"을 보는 것보다, **틀린 샘플(오분류)** 을 모아 보면 모델이 약한 패턴을 빠르게 파악할 수 있습니다.
- **어떻게 해석할까?**
  - 비슷한 클래스(예: `cat↔dog`, `automobile↔truck`)에서 헷갈리는지
  - 배경(하늘/풀/도로 등)이나 객체 크기/각도/가림(occlusion)에 민감한지
  - 데이터 증강(augmentation)이나 모델 크기/학습률 조절이 필요한지

아래 함수는 **오분류된 샘플만** 골라서 그려줍니다.


In [ ]:
# 오분류(실패) 샘플만 시각화
def plot_misclassified_samples(model, data_loader, max_samples: int = 10):
    """테스트 데이터에서 오분류된 샘플만 골라 시각화합니다.

    Args:
        model: 학습된 분류 모델
        data_loader: (data, target)을 내는 DataLoader
        max_samples: 최대 출력 샘플 수 (기본 10)
    """
    model.eval()
    mis_images, mis_labels, mis_preds = [], [], []

    with torch.no_grad():
        for data, target in data_loader:
            data = data.to(device)
            target = target.to(device)

            logits = model(data)                 # [B, num_classes] 로짓
            preds = logits.argmax(dim=1)         # 각 샘플의 예측 클래스

            wrong_mask = preds.ne(target)        # 오분류 위치만 True
            if wrong_mask.any():
                # 오분류된 샘플만 추출
                wrong_data = data[wrong_mask].detach().cpu()  # keep as Tensor for denormalize
                wrong_target = target[wrong_mask].detach().cpu()
                wrong_preds = preds[wrong_mask].detach().cpu()

                for x, y, p in zip(wrong_data, wrong_target, wrong_preds):
                    mis_images.append(x)
                    mis_labels.append(int(y))
                    mis_preds.append(int(p))
                    if len(mis_images) >= max_samples:
                        break

            if len(mis_images) >= max_samples:
                break

    if len(mis_images) == 0:
        print("오분류 샘플이 없습니다. (현재 설정/에폭에서 테스트를 모두 맞췄을 가능성이 있습니다)")
        return

    # 시각화 그리드: 최대 10개를 2×5로 보기 좋게 배치
    n = min(len(mis_images), max_samples)
    rows = 2 if n > 5 else 1
    cols = 5 if n > 5 else n

    fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3.2*rows))
    axes = np.array(axes).reshape(-1)  # axes를 1D로 평탄화

    for i in range(rows * cols):
        ax = axes[i]
        if i < n:
            img_chw = mis_images[i]  # CIFAR-10: (3,32,32)
            img_vis = denormalize_cifar10(img_chw).detach().cpu().numpy().transpose(1, 2, 0)  # HWC
            ax.imshow(img_vis)
            ax.set_title(f"GT: {class_names[mis_labels[i]]} / Pred: {class_names[mis_preds[i]]}")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


# 실행: 오분류 샘플만 출력
plot_misclassified_samples(model, test_loader, max_samples=10)


---
## 9) (추가 실습) ONNX Export + Netron으로 구조 확인하기

- **목적:** ViT를 ONNX로 내보내고, Netron에서 *토큰화(패치 임베딩) → Transformer 블록 → Head* 흐름을 그래프로 확인합니다.
- **관찰 포인트:**
  - `print(model)` / `torchinfo.summary(...)`에서 본 레이어가 Netron 그래프에서 어떻게 연결되는지
  - 입력 텐서 크기(예: `[B, 3, 32, 32]`)가 중간에서 토큰 시퀀스(`[B, N, D]`)로 변환되는 지점이 어디인지


In [ ]:
# 필요 패키지 확인 (이미 ensure_package가 있다면 그걸 써도 됩니다)
# - onnx / onnxruntime는 환경에 따라 없을 수 있으니, 없으면 설치하세요.
# ensure_package("onnx")
# ensure_package("onnxruntime")

import torch

onnx_path = "vit_cifar10.onnx"
dummy = torch.randn(1, 3, 32, 32, device=device)  # CIFAR-10 입력 형식 (RGB, 32x32)
model.eval()

try:
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        input_names=["input"],
        output_names=["logits"],
        opset_version=17,
        do_constant_folding=True,
        dynamic_axes={"input": {0: "B"}, "logits": {0: "B"}},  # 배치 크기만 가변 처리
    )
    print("Saved ONNX:", onnx_path)
    print("Netron에서 열어서 (patch embedding → transformer blocks → head) 흐름을 확인해보세요.")
except Exception as e:
    print("ONNX export failed:", e)
    print("힌트: onnx 패키지가 없다면 `pip install onnx`가 필요할 수 있습니다.")
